In [ ]:
import sys; sys.path.append('..')
import inflation, numpy as np, importlib, fd_validation, visualization, parametric_pillows, wall_generation
from numpy.linalg import norm
import MeshFEM, parallelism, benchmark, utils
import periodic_unit_helper
import numpy.linalg as la

In [ ]:
parallelism.set_max_num_tbb_threads(8)

In [ ]:
import importlib

In [ ]:
importlib.reload(parametric_pillows)

In [ ]:
triArea = 1

n_vx, n_edge = periodic_unit_helper.getBox(triArea)

In [ ]:
visualization.plot_line_segments(n_vx, n_edge)

In [ ]:
import importlib

In [ ]:
importlib.reload(parametric_pillows)

In [ ]:
from parametric_pillows import get_perioidic_mesh

In [ ]:
m, fuseMarkers, fuseSegments = wall_generation.triangulate_channel_walls(n_vx, n_edge, triArea, flags="Y")


In [ ]:
def is_bbox(point):
    if (point[0] == left_x or point[0] == right_x or point[1] == top_y or point[1] == bot_y):
        return True
    return False

In [ ]:
visualization.plot_2d_mesh(m, pointList=np.where(np.array(fuseMarkers) == 1)[0], width=5, height=5)

In [ ]:
ipu = inflation.InflatablePeriodicUnit(m, np.array(fuseMarkers) != 0, epsilon = 1e-5)

In [ ]:
import periodic_unit_helper

In [ ]:
fixedVars = periodic_unit_helper.get_center_fixedVars(ipu)

In [ ]:
ipu.periodicVolume()

In [ ]:
import py_newton_optimizer
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10

In [ ]:
from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(True)
viewer.show()

In [ ]:
viewer.showWireframe(True)

In [ ]:
Elastic = inflation.InflatableSheet.EnergyType.Elastic
Full = inflation.InflatableSheet.EnergyType.Full
Pressure = inflation.InflatableSheet.EnergyType.Pressure

In [ ]:
ipu.sheet.rigidMotionPinVars

In [ ]:
fd_perturb = np.random.uniform(-1e-3, 1e-3, ipu.numVars())

In [ ]:
ipu.setVars(ipu.getVars() + fd_perturb)

In [ ]:
ipu.setVars([1, 0, 1, 0, 0, 0, 0, 0, -1, 0, 0, 1])

In [ ]:
viewer.update()

In [ ]:
ipu.sheet.volume()

In [ ]:
ipu.periodicVolume()

In [ ]:
ipu.energy(energyType = Elastic)

In [ ]:
ipu.energy(energyType = Pressure)

In [ ]:
ipu.energy()

In [ ]:
import time, vis
benchmark.reset()
ipu.sheet.setUseTensionFieldEnergy(True)
ipu.sheet.setUseHessianProjectedEnergy(False)
ipu.sheet.pressure = 1
opts.niter = 200
framerate = 5 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb)
benchmark.report()

In [ ]:
viewer.update()

### Finite difference validation

In [ ]:
ipu.periodicVolume()

In [ ]:
ipu.sheet.volume()

In [ ]:
la.norm(ipu.gradientPeriodicPressurePotential())

In [ ]:
import periodic_unit_helper

In [ ]:
fd_perturb = np.random.uniform(-1e-3, 1e-3, ipu.numVars())

In [ ]:
# ipu.setVars(ipu.getVars() + fd_perturb)

In [ ]:
ipu.getVars()

In [ ]:
ipu_hessian = periodic_unit_helper.getNumpyArrayFromCSC(ipu.hessian(), reflect = True)

In [ ]:
isheet_hessian = periodic_unit_helper.getNumpyArrayFromCSC(ipu.sheet.hessian(),  reflect = True)

In [ ]:
A = periodic_unit_helper.getNumpyArrayFromCSC(ipu.getPeriodicPatchToInflatableSheetMapTranspose(),  reflect = False)

In [ ]:
la.norm(A @ ipu.sheet.gradient() - ipu.gradient())

In [ ]:
la.norm(A @ isheet_hessian @ A.transpose() - ipu_hessian)

In [ ]:
np.set_printoptions(suppress=True, precision = 4)

In [ ]:
ipu_hessian[:6, :6]

In [ ]:
isheet_hessian[:6, :6]

In [ ]:
(A[0] @ isheet_hessian) @ A[0]

In [ ]:
# A[:, :]

In [ ]:
(A @ isheet_hessian @ A.transpose())[:6, :6]

In [ ]:
np.set_printoptions(suppress=True, precision = 2)

In [ ]:
A.shape

In [ ]:
ipu.sheet.numVars()

In [ ]:
ipu.numVars()

In [ ]:



for i in range(6):
    A[:3, i * 3: (i + 1) * 3] = np.array([[1, 0, 0], 
                                         [1, 1, 0],
                                          [0, 1, 0]])

In [ ]:
A[:, :6]

In [ ]:
np.sign(A @ isheet_hessian @ A.transpose())

In [ ]:
ipu_hessian - np.sign(np.round(A @ isheet_hessian @ A.transpose()))

In [ ]:
ipu.energy(energyType = Pressure)

In [ ]:
ipu.sheet.pressure = 1

In [ ]:
fd_validation.gradConvergencePlot(ipu, customArgs = {"energyType": Pressure})

In [ ]:
fd_validation.hessConvergencePlot(ipu, customArgs = {"energyType": Pressure})

In [ ]:
fd_validation.gradConvergencePlot(ipu.sheet, customArgs = {"energyType": Pressure})

In [ ]:
fd_validation.hessConvergencePlot(ipu.sheet, customArgs = {"energyType": inflation.InflatableSheet.EnergyType.Pressure})

In [ ]:
viewer.update()

In [ ]:
class reduced_isheet_wrapper():
    def __init__(self, sheet, ipu):
        self.sheet = sheet
        self.vars = ipu.getVars()

    def setVars(self, v):
        self.sheet.setVars(A.transpose() @ v)
        self.vars = v
        
    def numVars(self):
        return A.shape[0]

    def getVars(self):
        return self.vars

    def energy(self):   return self.sheet.energy()
    def gradient(self): return A @ self.sheet.gradient()

In [ ]:
riw = reduced_isheet_wrapper(ipu.sheet, ipu)

In [ ]:
fd_validation.gradConvergencePlot(riw)

### Inflatable sheet

In [ ]:
isheet = inflation.InflatableSheet(m, np.array(fuseMarkers) != 0)

In [ ]:
import py_newton_optimizer
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10

In [ ]:
from tri_mesh_viewer import TriMeshViewer
sheet_viewer = TriMeshViewer(isheet, width=768, height=640)
sheet_viewer.showWireframe(True)
sheet_viewer.show()

In [ ]:
import time, vis
benchmark.reset()
isheet.setUseTensionFieldEnergy(True)
isheet.setUseHessianProjectedEnergy(False)
isheet.pressure = 1
opts.niter = 10
framerate = 5 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        sheet_viewer.update()
cr = inflation.inflation_newton(isheet, isheet.rigidMotionPinVars[:3], opts, callback=cb)
benchmark.report()

In [ ]:
sheet_viewer.update()

### Finite difference validation

In [ ]:
isheet.volume()

In [ ]:
fd_perturb = np.random.uniform(-1e-3, 1e-3, isheet.numVars())

In [ ]:
isheet.setVars(isheet.getVars() + fd_perturb)

In [ ]:
fd_validation.hessConvergencePlot(isheet, customArgs = {"energyType": inflation.InflatableSheet.EnergyType.Pressure})

### Validate gradient

In [ ]:
import fd_validation

In [ ]:
class fd_wrapper:
    def __init__(self, ipu):
        self.ipu = ipu

    def setVars(self, v):
        self.ipu.sheet.setVars(v)
    def numVars(self):
        return self.ipu.sheet.numVars()

    def getVars(self):
        return self.ipu.sheet.getVars()

    def energy(self):   return self.ipu.energyPeriodicPressurePotential()
    def gradient(self): return self.ipu.gradientPeriodicPressurePotential()

In [ ]:
fd_validation.gradConvergencePlot(fd_wrapper(ipu))